# Candlestick Data Visualization

Interactive candlestick charts using Plotly for trading data analysis.

## Import Libraries

In [18]:
import json
import os
import sys
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Add parent directory to path for imports
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

## Load Candlestick Data

Load data from JSON file. The file should contain OHLCV data in the format:
```json
[
  {"Date": "2020-01-01 17:00:00+0000", "Open": 1.12, "High": 1.13, "Low": 1.11, "Close": 1.12, "Volume": 0.0}
]
```

In [19]:
def load_candle_data(filename):
    """Load candlestick data from JSON file."""
    filepath = Path('../data') / filename
    
    if not filepath.exists():
        raise FileNotFoundError(f"File not found: {filepath}")
    
    with open(filepath, 'r') as f:
        data = json.load(f)
    
    df = pd.DataFrame(data)
    df['Date'] = pd.to_datetime(df['Date'], utc=True)
    df = df.sort_values('Date')
    
    print(f"Loaded {len(df)} candles from {filename}")
    print(f"Date range: {df['Date'].min()} to {df['Date'].max()}")
    print(f"Columns: {list(df.columns)}")
    
    return df

In [20]:
# List available data files
data_dir = Path('../data')
json_files = sorted([f.name for f in data_dir.glob('*.json')])

print("Available data files:")
for i, f in enumerate(json_files, 1):
    print(f"  {i}. {f}")

Available data files:
  1. candle_data_list_frxEURUSD_3600s.json


In [21]:
# Load your data file (change filename as needed)
filename = 'candle_data_list_frxEURUSD_3600s.json'  # 5-minute data

df = load_candle_data(filename)
df.head()

Loaded 18822 candles from candle_data_list_frxEURUSD_3600s.json
Date range: 2020-01-01 17:00:00+00:00 to 2023-01-02 00:00:00+00:00
Columns: ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']


,Date,Open,High,Low,Close,Volume
0,2020-01-01 17:00:00+00:00,1.12120,1.12121,1.12117,1.12120,0.0
1,2020-01-01 18:00:00+00:00,1.12106,1.12170,1.12106,1.12170,0.0
2,2020-01-01 19:00:00+00:00,1.12168,1.12218,1.12167,1.12182,0.0
3,2020-01-01 20:00:00+00:00,1.12182,1.12190,1.12157,1.12183,0.0
4,2020-01-01 21:00:00+00:00,1.12183,1.12244,1.12180,1.12205,0.0


## Plot Candlestick Chart

Create an interactive candlestick chart with volume subplot.

In [11]:
def plot_candlestick(df, title=None, show_volume=True, height=800):
    """
    Create interactive candlestick chart with optional volume subplot.
    
    Args:
        df: DataFrame with columns Date, Open, High, Low, Close, Volume
        title: Chart title (auto-generated if None)
        show_volume: Whether to show volume subplot
        height: Chart height in pixels
    """
    if title is None:
        symbol = "Unknown"
        date_range = f"{df['Date'].min().date()} to {df['Date'].max().date()}"
        title = f"Candlestick Chart - {len(df)} candles ({date_range})"
    
    # Create subplots
    if show_volume:
        fig = make_subplots(
            rows=2, cols=1,
            shared_xaxes=True,
            vertical_spacing=0.03,
            row_heights=[0.7, 0.3],
            subplot_titles=(title, 'Volume')
        )
    else:
        fig = go.Figure()
    
    # Add candlestick chart
    candlestick = go.Candlestick(
        x=df['Date'],
        open=df['Open'],
        high=df['High'],
        low=df['Low'],
        close=df['Close'],
        name='OHLC',
        increasing_line_color='green',
        decreasing_line_color='red'
    )
    
    if show_volume:
        fig.add_trace(candlestick, row=1, col=1)
        
        # Add volume bars
        colors = ['green' if close >= open else 'red' 
                  for close, open in zip(df['Close'], df['Open'])]
        
        volume = go.Bar(
            x=df['Date'],
            y=df['Volume'],
            name='Volume',
            marker_color=colors,
            showlegend=False
        )
        fig.add_trace(volume, row=2, col=1)
    else:
        fig.add_trace(candlestick)
        fig.update_layout(title=title)
    
    # Update layout
    fig.update_layout(
        height=height,
        xaxis_rangeslider_visible=False,
        hovermode='x unified',
        template='plotly_white'
    )
    
    # Update axes
    fig.update_xaxes(title_text="Date", row=2 if show_volume else 1, col=1)
    fig.update_yaxes(title_text="Price", row=1, col=1)
    if show_volume:
        fig.update_yaxes(title_text="Volume", row=2, col=1)
    
    return fig

In [13]:
# Plot the candlestick chart
fig = plot_candlestick(df, show_volume=True)
fig.show()

## Plot Subset of Data

Zoom in on a specific date range for detailed analysis.

In [14]:
# Filter to specific date range
start_date = df['Date'].min()
end_date = start_date + pd.Timedelta(hours=12)  # First 12 hours

df_subset = df[(df['Date'] >= start_date) & (df['Date'] <= end_date)]

print(f"Subset: {len(df_subset)} candles from {start_date} to {end_date}")

fig_subset = plot_candlestick(df_subset, title=f"Candlestick Chart (First 12 hours)", show_volume=True)
fig_subset.show()

Subset: 13 candles from 2020-01-01 17:00:00+00:00 to 2020-01-02 05:00:00+00:00


## Add Technical Indicators (Optional)

Add simple moving averages to the candlestick chart.

In [15]:
def plot_candlestick_with_ma(df, ma_periods=[20, 50], title=None, height=800):
    """
    Create candlestick chart with moving averages.
    
    Args:
        df: DataFrame with OHLCV data
        ma_periods: List of moving average periods
        title: Chart title
        height: Chart height in pixels
    """
    if title is None:
        title = f"Candlestick Chart with Moving Averages"
    
    fig = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.03,
        row_heights=[0.7, 0.3],
        subplot_titles=(title, 'Volume')
    )
    
    # Add candlestick
    fig.add_trace(
        go.Candlestick(
            x=df['Date'],
            open=df['Open'],
            high=df['High'],
            low=df['Low'],
            close=df['Close'],
            name='OHLC',
            increasing_line_color='green',
            decreasing_line_color='red'
        ),
        row=1, col=1
    )
    
    # Add moving averages
    colors = ['blue', 'orange', 'purple', 'brown']
    for i, period in enumerate(ma_periods):
        ma = df['Close'].rolling(window=period).mean()
        fig.add_trace(
            go.Scatter(
                x=df['Date'],
                y=ma,
                name=f'MA{period}',
                line=dict(color=colors[i % len(colors)], width=1.5)
            ),
            row=1, col=1
        )
    
    # Add volume
    colors = ['green' if close >= open else 'red' 
              for close, open in zip(df['Close'], df['Open'])]
    
    fig.add_trace(
        go.Bar(
            x=df['Date'],
            y=df['Volume'],
            name='Volume',
            marker_color=colors,
            showlegend=False
        ),
        row=2, col=1
    )
    
    # Update layout
    fig.update_layout(
        height=height,
        xaxis_rangeslider_visible=False,
        hovermode='x unified',
        template='plotly_white'
    )
    
    fig.update_xaxes(title_text="Date", row=2, col=1)
    fig.update_yaxes(title_text="Price", row=1, col=1)
    fig.update_yaxes(title_text="Volume", row=2, col=1)
    
    return fig

In [16]:
# Plot with moving averages
fig_ma = plot_candlestick_with_ma(df, ma_periods=[20, 50], title="Candlestick with MA20 and MA50")
fig_ma.show()

## Compare Multiple Timeframes

Load and compare data at different timeframes.

In [17]:
# Example: Load both 5m and 1h data if available
# Uncomment and adjust filenames as needed

# df_5m = load_candle_data('candle_data_list_frxEURUSD_300s.json')
# df_1h = load_candle_data('candle_data_list_frxEURUSD_3600s.json')

# fig_5m = plot_candlestick(df_5m, title="5-Minute Chart", height=400)
# fig_1h = plot_candlestick(df_1h, title="1-Hour Chart", height=400)

# fig_5m.show()
# fig_1h.show()

## Export Chart to HTML

Save the interactive chart as an HTML file for sharing.

In [ ]:
# Save chart to HTML file
output_dir = Path('../scripts/outputs')
output_dir.mkdir(exist_ok=True)

output_file = output_dir / 'candlestick_chart.html'
fig.write_html(str(output_file))

print(f"Chart saved to: {output_file}")